# OCEI™ v2.3 — Institutional Orbital Compute Underwriting, Cross-Model Reconciliation & Excel (.xlsx) Generator
Live zero-auth connection to [OrbitCompute.ai](https://orbitcompute.ai) (RFC-8785 + ECDSA P-256 attested, CERN/Zenodo Concept DOI: `10.5281/zenodo.24181722`).

**Outputs a 3-sheet Excel workbook (`OCEI_v2.3_Institutional_Audit.xlsx`):**
1. `Scenario_Reconciliation`: Canonical V3, Starship V4, Google Suncatcher (Trillium + New Glenn), Terafab 370K, Skeptic 0.6x Self-Build, 3.5%/9.0% Attrition, G3/G5 Storms & Live ERCOT/PJM/EPEX.
2. `Live_3Hub_Arbitrage`: Real-time ERCOT Texas, PJM Virginia & EPEX Europe spot prices vs. orbital breakeven.
3. `D1_24h_TimeSeries`: Empirical Cloudflare D1 15-minute snapshots & 30-day covenant availability.

In [ ]:
import hashlib, json, urllib.parse, urllib.request
from datetime import datetime, timezone
import pandas as pd

BASE = 'https://orbitcompute.ai'

def fetch_json(path, params=None):
    qs = ('?' + urllib.parse.urlencode(params)) if params else ''
    req = urllib.request.Request(BASE + path + qs, headers={'User-Agent': 'OCEI-Colab-Audit/2.3'})
    with urllib.request.urlopen(req, timeout=15) as r:
        return json.loads(r.read().decode('utf-8'))

def verify_rfc8785_sha256(payload):
    att = payload.get('attestation', {})
    unsigned = {k: v for k, v in payload.items() if k != 'attestation'}
    canon = json.dumps(unsigned, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode('utf-8')
    calc = hashlib.sha256(canon).hexdigest()
    return calc == att.get('state_hash_sha256'), att.get('key_id'), calc

live = fetch_json('/v1/index/ocei')
ok, kid, digest = verify_rfc8785_sha256(live)
print(f'RFC-8785 SHA-256 Verified: {ok} | Key: {kid} | Hash: {digest[:16]}...')

In [ ]:
SCENARIOS = [
    ('01_Canonical_100MW_Rubin_V3', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('02_Starship_V4_Block4_200t', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v4', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('03_Suncatcher_Trillium_NewGlenn', dict(power_mw=100, hw='trillium_tpu_v6e', launch_vehicle='new_glenn_9x4', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('04_Terafab_370K_95C_V4', dict(power_mw=100, hw='terafab_370k', launch_vehicle='starship_v4', loop_temp_c=95, grid_price_usd_mwh=115)),
    ('05_Scale_1GW_Hyperscale_V3', dict(power_mw=1000, hw='rubin_nvl72', launch_vehicle='starship_v3', loop_temp_c=80, grid_price_usd_mwh=115)),
    ('06_Skeptic_Earth_0.6x_SelfBuild', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', earth_capex_factor=0.6, grid_price_usd_mwh=115)),
    ('07_Skeptic_3.5pct_BurnIn_Attrition', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', annual_attrition_pct=3.5, grid_price_usd_mwh=115)),
    ('08_Skeptic_9.0pct_Stress_Attrition', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', annual_attrition_pct=9.0, grid_price_usd_mwh=115)),
    ('09_Live_Hub_ERCOT_Texas', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', grid_hub='ercot')),
    ('10_Live_Hub_PJM_Virginia', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', grid_hub='pjm')),
    ('11_Live_Hub_EPEX_Europe', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', grid_hub='epex')),
    ('12_Storm_G3_Solar_Event', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', storm_mode='g3_storm', grid_price_usd_mwh=115)),
    ('13_Storm_G5_Carrington_Extreme', dict(power_mw=100, hw='rubin_nvl72', launch_vehicle='starship_v3', storm_mode='g5_extreme', grid_price_usd_mwh=115))
]

rows = []
for label, params in SCENARIOS:
    d = fetch_json('/v1/index/ocei', params)
    oc, ic = d['ocei_standard'], d['input_configuration']
    sw, th = d['space_weather_and_young_daly_checkpointing'], d['thermodynamic_radiative_cooling']
    om, op = d['orbital_mechanics_and_launch_mass'], d['optical_oisl_laser_network']
    ec, cr = d['token_economics_and_5yr_tco'], d['credit_underwriting_and_covenants']
    rows.append({
        'Scenario': label,
        'OCEI_Spot': oc['ocei_spot_realtime_score'],
        'OCEI_Structural': oc['ocei_structural_5yr_score'],
        'Spread_bps': oc['net_orbital_spread_bps'],
        'Grid_USD_MWh': ic['terrestrial_grid_price_usd_mwh'],
        'Breakeven_USD_MWh': oc['breakeven_grid_usd_mwh'],
        'Dispatch': oc['arbitrage_dispatch_signal'],
        'Mass_kg_kW': om['deployed_mass_kg_per_it_kw'],
        'MMOD_Mass_t': om['mmod_shielded_total_constellation_mass_tons'],
        'Launches': om['starship_flights_required_at_125t'],
        'Net_Avail_pct': sw['net_effective_compute_availability_pct'],
        'Orbit_USD_MTok': ec['orbital_cost_per_1m_tokens_usd'],
        'Orbit_WACC_MTok': ec['wacc_adjusted_orbital_lcoc_usd_mtok'],
        'Earth_USD_MTok': ec['terrestrial_cost_per_1m_tokens_usd'],
        'Orbit_5yr_MUSD': ec['orbital_5yr_total_tco_musd'],
        'Earth_5yr_MUSD': ec['terrestrial_5yr_total_tco_musd'],
        'Savings_5yr_MUSD': ec['five_year_net_tco_savings_in_orbit_musd'],
        'Orbit_USD_W': cr['orbital_5yr_tco_usd_per_watt'],
        'Payback_Yrs': cr['implied_orbital_payback_years'],
        'OGS_Util_pct': op['ogs_downlink_utilization_pct'],
        'SHA256_Prefix': d['attestation']['state_hash_sha256'][:16]
    })

df_scenarios = pd.DataFrame(rows)
df_scenarios

In [ ]:
grids = fetch_json('/v1/telemetry/live-grids')
hist = fetch_json('/v1/telemetry/history', {'hours': 24, 'limit': 96})

df_grids = pd.DataFrame.from_dict(grids['live_terrestrial_spot_grids'], orient='index').reset_index(names='hub_id')
df_hist = pd.DataFrame(hist.get('snapshots', []))

out_file = f"OCEI_v2.3_Institutional_Audit_{datetime.now(timezone.utc).strftime('%Y-%m-%d')}.xlsx"
with pd.ExcelWriter(out_file) as writer:
    df_scenarios.to_excel(writer, sheet_name='Scenario_Reconciliation', index=False)
    df_grids.to_excel(writer, sheet_name='Live_3Hub_Arbitrage', index=False)
    df_hist.to_excel(writer, sheet_name='D1_24h_TimeSeries', index=False)

print(f'Generated 3-sheet Excel workbook: {out_file} ({len(df_hist)} D1 empirical snapshots included)')